### Label Studio Data Exports ###

In [1]:
import os
import sys
import glob
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Import this module with autoreload
%load_ext autoreload
%autoreload 2

import computervision as cv
from computervision.dentexdata import DentexData

# Print version info
print(f'Package version: {cv.__version__}')
print(f'Authors:         {cv.__authors__}')
print(f'Python version:  {sys.version}')

Package version: v0.0.2
Authors:         The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.11 (main, Sep 14 2025, 14:48:49) [GCC 13.3.0]


In [2]:
load_dotenv()
data_root = os.environ.get('DATA_DIR')
# Directory to store the data
image_dir = os.path.join(data_root, 'data_model', 'dataset_object_260101')

# Load annotations from Label Studio
export_dir = os.path.join(image_dir, 'export') 
annotations_file_name = 'result_coco.json'
annotations_file = os.path.join(export_dir, annotations_file_name)

# Load annotations
dtx = DentexData(data_dir=image_dir)
annotations = dtx.load_annotations(annotations_file)
print(f'Loaded annotations from file:\n{dtx.annotations_file}\n{annotations.keys()}')

Loaded annotations from file:
/home/andreas/data/data_model/dataset_object_260101/export/result_coco.json
dict_keys(['images', 'annotations', 'categories', 'info'])


In [3]:
im = pd.DataFrame(annotations.get('images')).\
    drop('path', axis=1).\
    rename(columns={'id': 'image_id'})
display(im)

ct = pd.DataFrame(annotations.get('categories')).\
    drop(['supercategory', 'metadata'], axis=1).\
    rename(columns={'id': 'category_id', 'name': 'pos'})         
display(ct)

an = pd.DataFrame(annotations.get('annotations')).\
    drop(['id', 'iscrowd'], axis=1)
display(an.head())

# Merge the annotations
df = im.merge(an, on='image_id', how='inner').\
    merge(ct, on='category_id', how='inner')
display(df.head())

,image_id,file_name,width,height
0,0,f83e9055-0cae21bb_20230320_14.png,1746,2422
1,1,29b02aef-0fb273c3_20220831_07.png,1336,2000
2,2,f8830d68-0fb273c3_20220831_10.png,2400,1708


,category_id,pos,color
0,1,01,#99906e
1,2,02,#72afb1
2,3,03,#b56bbf


,image_id,category_id,segmentation,area,bbox,annotator
0,0,1,[],6.132994e+05,"[450.10619469026574, 1611.0945615400135, 953.7...",1
1,0,2,[],1.016230e+06,"[264.3464333344806, 1003.8086979082487, 1481.6...",1
2,0,3,[],5.506433e+05,"[532.2684365781708, 589.4252473113412, 1213.73...",1
3,0,1,"[[610, 1855, 608, 1857, 608, 1862, 611, 1862, ...",2.100000e+01,"[608, 1855, 5, 8]",1
4,0,1,"[[508, 1716, 507, 1717, 507, 1718, 508, 1719]]",2.000000e+00,"[507, 1716, 2, 4]",1


,image_id,file_name,width,height,category_id,segmentation,area,bbox,annotator,pos,color
0,0,f83e9055-0cae21bb_20230320_14.png,1746,2422,1,[],6.132994e+05,"[450.10619469026574, 1611.0945615400135, 953.7...",1,01,#99906e
1,0,f83e9055-0cae21bb_20230320_14.png,1746,2422,2,[],1.016230e+06,"[264.3464333344806, 1003.8086979082487, 1481.6...",1,02,#72afb1
2,0,f83e9055-0cae21bb_20230320_14.png,1746,2422,3,[],5.506433e+05,"[532.2684365781708, 589.4252473113412, 1213.73...",1,03,#b56bbf
3,0,f83e9055-0cae21bb_20230320_14.png,1746,2422,1,"[[610, 1855, 608, 1857, 608, 1862, 611, 1862, ...",2.100000e+01,"[608, 1855, 5, 8]",1,01,#99906e
4,0,f83e9055-0cae21bb_20230320_14.png,1746,2422,1,"[[508, 1716, 507, 1717, 507, 1718, 508, 1719]]",2.000000e+00,"[507, 1716, 2, 4]",1,01,#99906e


In [5]:
image_id_list = sorted(list(df['image_id'].unique()))
idx = 0
image_id = image_id_list[idx]
df_idx = df.loc[df['image_id'] == image_id].reset_index(drop=True)
file_base = df_idx['file_name'].unique()[0].split('-', maxsplit=1)[1]
file = glob.glob(os.path.join(image_dir, f'*{file_base}'))[0]
print(image_dir)
print(file)



/home/andreas/data/data_model/dataset_object_260101
/home/andreas/data/data_model/dataset_object_260101/0cae21bb_20230320_14.png


f8830d68-0fb273c3_20220831_10.png
/home/andreas/data/data_model/dataset_object_260101
